# Training a microWakeWord Model

This notebook steps you through training a basic microWakeWord model. It is intended as a **starting point** for advanced users. You should use Python 3.10.

**The model generated will most likely not be usable for everyday use; it may be difficult to trigger or falsely activates too frequently. You will most likely have to experiment with many different settings to obtain a decent model!**

In the comment at the start of certain blocks, I note some specific settings to consider modifying.

This runs on Google Colab, but is extremely slow compared to training on a local GPU. If you must use Colab, be sure to Change the runtime type to a GPU. Even then, it still slow!

At the end of this notebook, you will be able to download a tflite file. To use this in ESPHome, you need to write a model manifest JSON file. See the [ESPHome documentation](https://esphome.io/components/micro_wake_word) for the details and the [model repo](https://github.com/esphome/micro-wake-word-models/tree/main/models/v2) for examples.


**Data licensing status:** The augmentation and negative-audio steps below select documented replacement sources and export attribution records. The legacy Piper `.pt` model and product dependencies still require a separate review. This notebook is not a blanket commercial-use clearance.

Data preparation downloads approximately 15 GB of archives; allow at least 40 GB of free disk space. It requires the system `zip` command for the multipart FSD50K archive. See [the data preparation guide](../documentation/licensed_training_data.md).


In [ ]:
# Installs microWakeWord. Be sure to restart the session after this is finished.
import platform

if platform.system() == "Darwin":
    # `pymicro-features` is installed from a fork to support building on macOS
    !pip install 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

# `audio-metadata` is installed from a fork to unpin `attrs` from a version that breaks Jupyter
!pip install 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'

# Install this updated checkout, which includes the licensed-data preparation code.
from pathlib import Path

repo_path = next((path for path in (Path.cwd(), Path.cwd().parent, Path.cwd() / "microWakeWord")
                  if (path / "microwakeword/audio/licensed_data.py").is_file()), None)
if repo_path is None:
    raise FileNotFoundError("Upload or clone this updated checkout first; see licensed_training_data.md")
%pip install -e "{repo_path}"

In [ ]:
# Generates 1 sample of the target word for manual verification.

target_word = 'khum_puter'  # Phonetic spellings may produce better samples

import os
import sys

from IPython.display import Audio

if not os.path.exists("./piper-sample-generator"):
    !git clone https://github.com/rhasspy/piper-sample-generator

# Use upstream master (also on macOS); switch any old mps-support checkout first.
%pip install -e ./piper-sample-generator

if not os.path.exists("piper-sample-generator/models/en_US-libritts_r-medium.pt"):
    !wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

# Run from the checkout so the generator can import piper_train.
!cd piper-sample-generator && "{sys.executable}" -m piper_sample_generator "{target_word}" \
--model models/en_US-libritts_r-medium.pt \
--max-samples 1 \
--batch-size 1 \
--output-dir ../generated_samples

Audio("generated_samples/0.wav", autoplay=True)

In [ ]:
# Generates a larger amount of wake word samples.
# Start here when trying to improve your model.
# See https://github.com/rhasspy/piper-sample-generator for the full set of
# parameters. In particular, experiment with noise-scales and noise-scale-ws,
# generating negative samples similar to the wake word, and generating many more
# wake word samples, possibly with different phonetic pronunciations.

!cd piper-sample-generator && "{sys.executable}" -m piper_sample_generator "{target_word}" \
--model models/en_US-libritts_r-medium.pt \
--max-samples 1000 \
--batch-size 100 \
--output-dir ../generated_samples

In [ ]:
# Prepare replacement augmentation and negative audio, with per-file provenance.
# First run downloads ~15 GB; later runs verify the prepared audio hashes.
# Only CC0/CC-BY FSD50K clips and simulated OpenSLR 28 RIRs are selected.
# LibriSpeech supplies the replacement speech negatives.
# TTS provenance and product dependencies remain outside this data audit.

from microwakeword.audio.licensed_data import prepare_licensed_data

data_root = prepare_licensed_data("licensed_data_v1")
print(f"Audio: {data_root}")
print(f"License manifest: {data_root / 'manifest.csv'}")
print(f"Attribution: {data_root / 'ATTRIBUTION.md'}")


In [ ]:
# Sets up the augmentations.
# To improve your model, experiment with these settings and use more sources of
# background clips.

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

clips = Clips(input_directory='generated_samples',
              file_pattern='*.wav',
              max_clip_duration_s=None,
              remove_silence=False,
              random_split_seed=10,
              split_count=0.1,
              )
# Separate background pools keep held-out audio out of training augmentation.
augmenters = {
    split: Augmentation(
        augmentation_duration_s=3.2,
        augmentation_probabilities={
            "SevenBandParametricEQ": 0.1,
            "TanhDistortion": 0.1,
            "PitchShift": 0.1,
            "BandStopFilter": 0.1,
            "AddColorNoise": 0.1,
            "AddBackgroundNoise": 0.75,
            "Gain": 1.0,
            "RIR": 0.5,
        },
        impulse_paths=[str(data_root / "rirs")],
        background_paths=[str(data_root / "noise" / split)],
        background_min_snr_db=-5,
        background_max_snr_db=10,
        min_jitter_s=0.195,
        max_jitter_s=0.205,
    )
    for split in ("training", "validation", "testing")
}
augmenter = augmenters["training"]


In [ ]:
# Augment a random clip and play it back to verify it works well

from IPython.display import Audio
from microwakeword.audio.audio_utils import save_clip

random_clip = clips.get_random_clip()
augmented_clip = augmenter.augment_clip(random_clip)
save_clip(augmented_clip, 'augmented_clip.wav')

Audio("augmented_clip.wav", autoplay=True)

In [ ]:
# Augment samples and save the training, validation, and testing sets.
# Validating and testing samples generated the same way can make the model
# benchmark better than it performs in real-word use. Use real samples or TTS
# samples generated with a different TTS engine to potentially get more accurate
# benchmarks.

import os
from mmap_ninja.ragged import RaggedMmap

output_dir = 'licensed_features_v1/positive'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

splits = ["training", "validation", "testing"]
for split in splits:
  out_dir = os.path.join(output_dir, split)
  if not os.path.exists(out_dir):
      os.mkdir(out_dir)


  split_name = "train"
  repetition = 2

  spectrograms = SpectrogramGeneration(clips=clips,    # Window length is 30 ms
                                     augmenter=augmenters[split],
                                     slide_frames=10,    # Uses the same spectrogram repeatedly, just shifted over by one frame. This simulates the streaming inferences while training/validating in nonstreaming mode.
                                     step_ms=10,
                                     )
  if split == "validation":
    split_name = "validation"
    repetition = 1
  elif split == "testing":
    split_name = "test"
    repetition = 1
    spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenters[split],
                                     slide_frames=1,    # The testing set uses the streaming version of the model, so no artificial repetition is necessary
                                     step_ms=10,
                                     )

  RaggedMmap.from_generator(
      out_dir=os.path.join(out_dir, 'wakeword_mmap'),
      sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
      batch_size=100,
      verbose=True,
  )

In [ ]:
# Build replacement negative features from the audited raw audio.
# Keep every source split intact. Ambient evaluation uses original clips >=6s,
# without synthetic repetition; it is a proxy for real-device room recordings.

import json
from pathlib import Path
from mmap_ninja.ragged import RaggedMmap
from microwakeword.audio.licensed_data import digest

negative_root = Path("licensed_features_v1/negative")
for corpus, pattern in (("speech", "*.flac"), ("noise", "*.wav")):
    for split in ("training", "validation", "testing"):
        source_dir = data_root / corpus / split
        negative_clips = Clips(
            input_directory=str(source_dir),
            file_pattern=pattern,
            repeat_clip_min_duration_s=6,
        )
        generator = SpectrogramGeneration(
            clips=negative_clips,
            step_ms=10,
            split_spectrogram_duration_s=5,
        )
        out_dir = negative_root / corpus / split
        out_dir.mkdir(parents=True, exist_ok=True)
        RaggedMmap.from_generator(
            out_dir=str(out_dir / "negative_mmap"),
            sample_generator=generator.spectrogram_generator(),
            batch_size=100,
            verbose=True,
        )
        if split != "training":
            ambient_clips = Clips(
                input_directory=str(source_dir),
                file_pattern=pattern,
                min_clip_duration_s=6,
            )
            if len(ambient_clips.clips) == 0:
                raise ValueError(f"No original clips >=6s for ambient evaluation: {source_dir}")
            ambient_generator = SpectrogramGeneration(clips=ambient_clips, step_ms=10)
            out_dir = negative_root / "ambient" / f"{split}_ambient"
            out_dir.mkdir(parents=True, exist_ok=True)
            RaggedMmap.from_generator(
                out_dir=str(out_dir / f"{corpus}_mmap"),
                sample_generator=ambient_generator.spectrogram_generator(),
                batch_size=100,
                verbose=True,
            )

# Link derived features to the exact original-file license manifest.
lineage = {
    "audio_manifest": str((data_root / "manifest.json").resolve()),
    "audio_manifest_sha256": digest(data_root / "manifest.json"),
    "negative_features": str(negative_root),
    "positive_features": "licensed_features_v1/positive",
    "processing": "16 kHz mono on access; 10 ms feature step; 5s negative chunks; "
                  "short classification negatives repeated to >=6s; ambient clips not repeated",
    "positive_augmentation": {split: {
        "background_pool": str(data_root / "noise" / split),
        "rir_pool": str(data_root / "rirs"),
    } for split in ("training", "validation", "testing")},
    "pending": "Legacy TTS provenance, dependency audit and real-device evaluation",
}
(negative_root.parent / "feature_sources.json").write_text(json.dumps(lineage, indent=2) + "\n")


In [ ]:
# Save a yaml config that controls the training process
# These hyperparamters can make a huge different in model quality.
# Experiment with sampling and penalty weights and increasing the number of
# training steps.

import yaml
import os

config = {}

config["window_step_ms"] = 10

config["train_dir"] = (
    "trained_models/wakeword_licensed_inputs"
)


# Each feature_dir should have at least one of the following folders with this structure:
#  training/
#    ragged_mmap_folders_ending_in_mmap
#  testing/
#    ragged_mmap_folders_ending_in_mmap
#  testing_ambient/
#    ragged_mmap_folders_ending_in_mmap
#  validation/
#    ragged_mmap_folders_ending_in_mmap
#  validation_ambient/
#    ragged_mmap_folders_ending_in_mmap
#
#  sampling_weight: Weight for choosing a spectrogram from this set in the batch
#  penalty_weight: Penalizing weight for incorrect predictions from this set
#  truth: Boolean whether this set has positive samples or negative samples
#  truncation_strategy = If spectrograms in the set are longer than necessary for training, how are they truncated
#       - random: choose a random portion of the entire spectrogram - useful for long negative samples
#       - truncate_start: remove the start of the spectrogram
#       - truncate_end: remove the end of the spectrogram
#       - split: Split the longer spectrogram into separate spectrograms offset by 100 ms. Only for ambient sets

config["features"] = [
    {
        "features_dir": "licensed_features_v1/positive",
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truth": True,
        "truncation_strategy": "truncate_start",
        "type": "mmap",
    },
    {
        "features_dir": "licensed_features_v1/negative/speech",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "licensed_features_v1/negative/noise",
        "sampling_weight": 5.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    { # Only used for validation and testing
        "features_dir": "licensed_features_v1/negative/ambient",
        "sampling_weight": 0.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "split",
        "type": "mmap",
    },
]



# Penalizing weight for incorrect class predictions - lists that correspond to training steps
config["positive_class_weight"] = [1]
config["negative_class_weight"] = [20]

# Number of training steps in each iteration - various other settings are configured as lists that corresponds to different steps
config["training_steps"] = [10000]
config["learning_rates"] = [0.001]  # Learning rates for Adam optimizer - list that corresponds to training steps
config["batch_size"] = 128

config["time_mask_max_size"] = [0]  # SpecAugment - list that corresponds to training steps
config["time_mask_count"] = [0]  # SpecAugment - list that corresponds to training steps
config["freq_mask_max_size"] = [0]  # SpecAugment - list that corresponds to training steps
config["freq_mask_count"] = [0]  # SpecAugment - list that corresponds to training steps

config["eval_step_interval"] = (500)  # Test the validation sets after every this many steps
config["clip_duration_ms"] = (1500)  # Maximum length of wake word that the streaming model will accept


# The best model weights are chosen first by minimizing the specified minimization metric below the specified target_minimization
# Once the target has been met, it chooses the maximum of the maximization metric. Set 'minimization_metric' to None to only maximize
# Available metrics:
#   - "loss" - cross entropy error on validation set
#   - "accuracy" - accuracy of validation set
#   - "recall" - recall of validation set
#   - "precision" - precision of validation set
#   - "false_positive_rate" - false positive rate of validation set
#   - "false_negative_rate" - false negative rate of validation set
#   - "ambient_false_positives" - count of false positives from the split validation_ambient set
#   - "ambient_false_positives_per_hour" - estimated number of false positives per hour on the split validation_ambient set
config["target_minimization"] = 0.9
config["minimization_metric"] = None  # Set to None to disable

config["maximization_metric"] = "average_viable_recall"

with open(os.path.join("training_parameters_licensed_inputs.yaml"), "w") as file:
    documents = yaml.dump(config, file)

In [ ]:
# Trains a model. When finished, it will quantize and convert the model to a
# streaming version suitable for on-device detection.
# It will resume if stopped, but it will start over at the configured training
# steps in the yaml file.
# Change --train 0 to only convert and test the best-weighted model.
# On Google colab, it doesn't print the mini-batch results, so it may appear
# stuck for several minutes! Additionally, it is very slow compared to training
# on a local GPU.

!python -m microwakeword.model_train_eval \
--training_config='training_parameters_licensed_inputs.yaml' \
--train 1 \
--restore_checkpoint 1 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block  "1, 1, 1, 1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

In [ ]:
# Downloads the tflite model file. To use on the device, you need to write a
# Model JSON file. See https://esphome.io/components/micro_wake_word for the
# documentation and
# https://github.com/esphome/micro-wake-word-models/tree/main/models/v2 for
# examples. Adjust the probability threshold based on the test results obtained
# after training is finished. You may also need to increase the Tensor arena
# model size if the model fails to load.

from google.colab import files

files.download(f"trained_models/wakeword_licensed_inputs/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite")